In [ ]:
from typing import List, TypedDict
import time

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

In [ ]:
docs = (
    PyPDFLoader("./documents/book1.pdf").load() +
    PyPDFLoader("./documents/book2.pdf").load() +
    PyPDFLoader("./documents/book3.pdf").load()
)

In [ ]:
# 4) LLM + prompt
llm = ChatOllama(model="qwen3:1.7b", temperature=0)

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=150
)

chunks = splitter.split_documents(docs)


# Clean invalid Unicode characters
for chunk in chunks:
    chunk.page_content = (
        chunk.page_content
        .encode("utf-8", errors="replace")
        .decode("utf-8")
    )


# Embed in batches
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

BATCH_SIZE = 50

vector_store = None

for i in range(0, len(chunks), BATCH_SIZE):

    batch = chunks[i:i + BATCH_SIZE]

    print(
        f"Embedding chunks "
        f"{i + 1}-{min(i + BATCH_SIZE, len(chunks))} "
        f"of {len(chunks)}"
    )

    batch_vector_store = FAISS.from_documents(
        batch,
        embeddings
    )

    if vector_store is None:
        vector_store = batch_vector_store
    else:
        vector_store.merge_from(batch_vector_store)


retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

In [ ]:
class State(TypedDict):
    question: str
    docs: List[Document]
    answer: str

In [ ]:
def retrieve(state):
    q = state["question"]
    return {"docs": retriever.invoke(q)}

In [ ]:

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer only from the context. If not in context, say you don't know."),
        ("human", "Question: {question}\n\nContext:\n{context}"),
    ]
)
def generate(state):
    context = "\n\n".join(d.page_content for d in state["docs"])
    out = (prompt | llm).invoke({"question": state["question"], "context": context})
    return {"answer": out.content}


In [ ]:
g = StateGraph(State)
g.add_node("retrieve", retrieve)
g.add_node("generate", generate)
g.add_edge(START, "retrieve")
g.add_edge("retrieve", "generate")
g.add_edge("generate", END)
app = g.compile()

app

In [ ]:
# 5) Run
res = app.invoke({"question": "WHat is a transformer in deep learning.", "docs": [], "answer": ""})
print(res["answer"])

In [ ]:
print(res['docs'][0].page_content)
print('*'*100)
print(res['docs'][1].page_content)
print('*'*100)
print(res['docs'][2].page_content)
print('*'*100)
print(res['docs'][3].page_content)